# HolidaysPrep — exploration

Pour chaque hôtel (coords RodPrep) et chaque **année × mois** :

- `nb_jours_feries` — jours fériés légaux
- `nb_jours_vacances_scolaires` — jours de vacances (zone A/B/C)
- `nb_jours_vacances_hors_feries` — vacances **hors** jours fériés

Export Excel dans `../Output/holidays_monthly.xlsx`.

> Après mise à jour du code : **Kernel → Restart & Run All**.

In [ ]:
import requests
from datetime import datetime

# ===================== CONFIG =====================
# Table de correspondance Département → Zone (mise à jour 2026)
DEPARTEMENT_TO_ZONE = {
    # Zone A
    '01','03','07','15','21','25','26','38','39','42','43','58','63','69','70','73','74','89','90': 'A',
    # Zone B
    '02','04','05','06','08','09','11','12','13','14','16','17','18','19','22','23','24','27','28','29','30',
    '31','32','34','35','36','37','40','44','45','46','47','48','49','50','51','52','53','54','55','56','57',
    '59','60','61','62','64','65','66','67','68','71','72','75','76','77','78','79','80','81','82','83','84',
    '85','86','87','88','91','92','93','94','95': 'B',  # Note: certains sont C, mais on ajuste ci-dessous
    # Zone C (prioritaire)
    '75','77','78','91','92','93','94','95': 'C',   # Île-de-France
    '31','34','59','69': 'B',  # ajustements
    # Corse
    '2A','2B': 'Corse',
}

def get_zone_from_coords(lat: float, lon: float):
    """Retourne la zone scolaire à partir de lat/long"""
    try:
        # 1. Géocodage inverse avec l'API officielle
        url = f"https://api-adresse.data.gouv.fr/reverse/?lat={lat}&lon={lon}&limit=1"
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        if not data.get('features'):
            return None, "Aucune commune trouvée"

        props = data['features'][0]['properties']
        code_dep = props.get('codeDepartement') or props.get('context', '').split(',')[-2].strip()
        
        if not code_dep:
            return None, "Impossible de déterminer le département"

        # Nettoyage
        code_dep = code_dep.zfill(2).upper()

        # Recherche de la zone
        zone = DEPARTEMENT_TO_ZONE.get(code_dep)
        if not zone:
            # Fallback : on regarde les premiers chiffres
            zone = DEPARTEMENT_TO_ZONE.get(code_dep[:2], 'B')  # par défaut B

        commune = props.get('city', props.get('name', 'Inconnue'))
        return zone, f"{commune} ({code_dep})"

    except Exception as e:
        return None, f"Erreur géocodage: {e}"


def get_vacances_for_zone(zone: str):
    """Récupère les vacances pour une zone via l'API officielle"""
    try:
        # API data.education.gouv.fr
        url = "https://data.education.gouv.fr/api/records/1.0/search/"
        params = {
            "dataset": "fr-en-calendrier-scolaire",
            "rows": 100,
            "sort": "-start_date",
            "refine.zones": f"Zone {zone}" if zone != "Corse" else "Corse",
        }
        
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        records = resp.json().get('records', [])

        periodes = []
        for record in records:
            fields = record['fields']
            description = fields.get('description', '')
            start = fields.get('start_date')
            end = fields.get('end_date')
            
            if start and end:
                periodes.append({
                    'periode': description,
                    'du': start[:10],
                    'au': end[:10]
                })

        return sorted(periodes, key=lambda x: x['du'])

    except Exception as e:
        return [{"erreur": str(e)}]


# ===================== UTILISATION =====================

if __name__ == "__main__":
    # Exemple
    lat = 48.8566   # Paris
    lon = 2.3522

    print(f"Recherche pour lat={lat}, lon={lon}...\n")
    
    zone, location = get_zone_from_coords(lat, lon)
    print(f"📍 Localisation : {location}")
    print(f"🧭 Zone scolaire : {zone}\n")

    if zone:
        vacances = get_vacances_for_zone(zone)
        print("🎒 Périodes de vacances scolaires :")
        for v in vacances:
            print(f"• {v['periode']}: du {v['du']} au {v['au']}")
    else:
        print("Impossible de déterminer la zone.")from pathlib import Path
import sys

import pandas as pd

HERE = Path.cwd().resolve()
PROJECT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "prepare" / "holidays_prep").is_dir() and (candidate / "rod_ia").is_dir():
        PROJECT = candidate
        break
if PROJECT is None:
    raise RuntimeError(f"Racine projet introuvable depuis {HERE}")

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

for mod_name in list(sys.modules):
    if mod_name == "prepare" or mod_name.startswith("prepare."):
        del sys.modules[mod_name]

from prepare.paths import default_paths
from prepare.holidays_prep import (
    HolidaysPrep,
    SchoolHolidayCalendar,
    french_public_holidays,
    resolve_zone_from_coords,
    fetch_school_holidays,
)
from prepare.holidays_prep.calendar import DEPARTEMENT_TO_ZONE

paths = default_paths()
INPUT_DIR = paths.holidays_input
OUTPUT_DIR = paths.holidays_output
ROD_OUTPUT = paths.rod_output

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)

print("PROJECT:", PROJECT)
print("Zones connues (ex.):", {k: DEPARTEMENT_TO_ZONE[k] for k in ("06", "75", "69", "67")})

## 1. Zone scolaire depuis des coordonnées (ex. Paris)

In [ ]:
lat, lon = 48.8566, 2.3522  # Paris
geo = resolve_zone_from_coords(lat, lon)
print(geo)

print("\nJours fériés 2025 (extrait):")
for d in sorted(french_public_holidays(2025, departement=geo.departement))[:8]:
    print(" -", d)

## 2. Vacances scolaires de la zone (API education.gouv)

In [ ]:
periodes = fetch_school_holidays(geo.zone, years=(2024, 2025, 2026))
print(f"Zone {geo.zone} — {len(periodes)} périodes")
for p in periodes[-8:]:
    print(f"• {p.description}: {p.start} → {p.end} (reprise exclue)")

## 3. Entrée hôtels depuis RodPrep

In [ ]:
TARGET_YEARS = (2023, 2024, 2025, 2026)

prep = HolidaysPrep(INPUT_DIR, OUTPUT_DIR, target_years=TARGET_YEARS)
prep.fill_input_from_rod(ROD_OUTPUT)
hotels = prep.load_input()
print(f"Hôtels : {len(hotels)}")
hotels

## 4. Pipeline complet → Excel `Output/`

Grain : `hotel_code × annee × mois`.

Colonnes principales : `nb_jours_feries`, `nb_jours_vacances_scolaires`, `nb_jours_vacances_hors_feries`.

In [ ]:
frame = prep.run()
print("shape:", frame.shape)
print("zones:", frame.groupby("hotel_code")["zone_scolaire"].first().to_dict())
print("Excel:", OUTPUT_DIR / "holidays_monthly.xlsx")

frame[
    [
        "hotel_code",
        "zone_scolaire",
        "annee",
        "mois",
        "nb_jours_feries",
        "nb_jours_vacances_scolaires",
        "nb_jours_vacances_hors_feries",
    ]
].head(24)

## 5. Aperçu Excel (résumé annuel)

In [ ]:
resume = pd.read_excel(OUTPUT_DIR / "holidays_monthly.xlsx", sheet_name="resume_annuel")
resume